<a href="https://colab.research.google.com/github/gez2code/dermamnist-hybrid-study/blob/main/main_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# 1. Install the necessary libraries
# - wandb: for experiment tracking
# - medmnist: for the dataset
!pip install -q wandb medmnist

# 2. Import libraries
import wandb
from google.colab import userdata # This reads the secret key
import os

# 3. Login to WandB securely
# This grabs the key you saved in the "Secrets" tab
wandb_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_key)

# 4. Initialize the Project
# This creates a project in your WandB dashboard
run = wandb.init(
    project="DermaMNIST-Hybrid-Project",
    name="Setup-Test-Run",
    notes="Testing if the lab environment works."
)

# 5. Log a dummy metric to make sure it works
wandb.log({"accuracy": 0.5, "loss": 1.2})
print("✅ Setup Complete! Check your WandB dashboard.")

# 6. Finish the run
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


✅ Setup Complete! Check your WandB dashboard.


accuracy,▁
loss,▁
accuracy,0.5
loss,1.2


In [18]:
# ==========================================
# PHASE 2: DATA PIPELINE (Adapted for DermaMNIST)
# ==========================================
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
import medmnist
from medmnist import DermaMNIST

# Check if we are ready
print(f"MedMNIST v{medmnist.__version__} @ {medmnist.__file__}")

# --- 1. Define Transforms ---

# "Original" (28x28) - The native size of this dataset
transform_original = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# "Midsize" (96x96) - UPSCALED from 28x28 (Blurs the image slightly)
transform_midsize = transforms.Compose([
    transforms.Resize((96, 96)), # We must Resize, not Crop, because source is tiny
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# "Large" (224x224) - UPSCALED (Standard for ResNet-18 ImageNet weights)
transform_large = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# --- 2. Create Datasets (Train, Val, Test) ---
# DermaMNIST handles the downloading automatically, just like that Kaggle notebook
datasets = {}

for split in ['train', 'val', 'test']:
    # Standard Mode (28x28)
    datasets[f'orig_{split}'] = DermaMNIST(split=split, transform=transform_original, download=True)

    # Upscaled Modes
    datasets[f'mid_{split}']  = DermaMNIST(split=split, transform=transform_midsize, download=True)
    datasets[f'large_{split}'] = DermaMNIST(split=split, transform=transform_large, download=True)

# --- 3. Master Loader Dictionary ---
all_loaders = {
    # Original (28x28)
    'original':      DataLoader(datasets['orig_train'], batch_size=64, shuffle=True),
    'val_original':  DataLoader(datasets['orig_val'],   batch_size=64, shuffle=False),
    'test_original': DataLoader(datasets['orig_test'],  batch_size=64, shuffle=False),

    # Midsize (96x96 - Upscaled)
    'midsize':       DataLoader(datasets['mid_train'],  batch_size=64, shuffle=True),
    'val_midsize':   DataLoader(datasets['mid_val'],    batch_size=64, shuffle=False),
    'test_midsize':  DataLoader(datasets['mid_test'],   batch_size=64, shuffle=False),

    # Large (224x224 - Upscaled)
    'large':         DataLoader(datasets['large_train'], batch_size=64, shuffle=True),
    'val_large':     DataLoader(datasets['large_val'],   batch_size=64, shuffle=False),
    'test_large':    DataLoader(datasets['large_test'],  batch_size=64, shuffle=False)
}

# Important: Update Global Class Count (DermaMNIST has 7 classes)
n_classes = 7

print("✅ DermaMNIST Data Pipeline Ready: Loaded Train, Val, and Test splits.")
print(f"   Shape Check (Original): {datasets['orig_train'][0][0].shape}")

MedMNIST v3.0.2 @ /usr/local/lib/python3.12/dist-packages/medmnist/__init__.py
✅ DermaMNIST Data Pipeline Ready: Loaded Train, Val, and Test splits.
   Shape Check (Original): torch.Size([3, 28, 28])


In [19]:
# ==========================================
# PHASE 3: DYNAMIC TRAINING ENGINE (Final)
# ==========================================
import torch
import torch.optim as optim
import torch.nn as nn
import gc
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Engine running on: {device}")

def train_dynamic(model, model_name, loaders_dict, epochs=10, loader_key=None,
                  learning_rate=0.001, weight_decay=0.0,
                  extra_config=None):
    """
    Args:
        extra_config (dict): Optional dictionary of architectural details to log.
    """

    # --- STEP 1: INTELLIGENT DATA SELECTION ---
    if loader_key is not None:
        train_loader = loaders_dict[loader_key]
        test_loader  = loaders_dict[f"val_{loader_key}"]
    else:
        # Fallback Logic
        train_loader = loaders_dict['original']
        test_loader  = loaders_dict['val_original']

    # --- STEP 2: TRACKING ---
    img_h = train_loader.dataset[0][0].shape[1]

    # Base Config
    run_config = {
        "architecture": model_name,
        "epochs": epochs,
        "resolution": f"{img_h}x{img_h}",
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "split_used": "Train + Validation"
    }

    # Merge with Custom Config (The Magic Fix)
    if extra_config:
        run_config.update(extra_config)

    wandb.init(
        project="DermaMNIST-Hybrid-Project",
        name=f"{model_name}",
        config=run_config,
        reinit=True
    )

    # --- STEP 3: SETUP ---
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    print(f"🚀 Training {model_name} | LR: {learning_rate} | Reg: {weight_decay}")

    # --- STEP 4: TRAINING LOOP ---
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.squeeze().long()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        acc = correct / total
        loss_avg = running_loss / len(train_loader)

        wandb.log({"train_loss": loss_avg, "train_acc": acc})
        print(f"   [Epoch {epoch+1}/{epochs}] Loss: {loss_avg:.4f} | Acc: {acc:.4f}")

    # --- STEP 5: VALIDATION ---
    print(f"📝 Validating {model_name}...")
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.squeeze().long()
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    print(f"🏆 Final Validation Accuracy: {val_acc:.2%}")

    wandb.log({"val_accuracy": val_acc})
    wandb.finish()

    return val_acc

def reset_memory(model=None):
    print("\n🧹 Cleaning up GPU memory...")
    if model is not None:
        try: del model
        except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("✨ Memory Clean.\n")

⚙️ Engine running on: cuda


In [21]:
# ==========================================
# PHASE 3.1: THE UNIVERSAL DYNAMIC ARCHITECTURE
# ==========================================
import torch.nn as nn

class DynamicCNN(nn.Module):
    def __init__(self, num_classes, input_size,
                 base_filters=16,
                 dropout_rate=0.0,
                 kernel_size=3,
                 num_layers=2,
                 stride=1,
                 pool_type='max'):
        """
        The Universal CNN Class.
        Args:
            num_layers (int): How many conv blocks to stack.
            stride (int): Convolutions step size (1=Standard, 2=Downsample).
            pool_type (str): 'max', 'avg', or None.
        """
        super(DynamicCNN, self).__init__()

        self.layers = nn.ModuleList()

        # 1. SETUP VARIABLES
        p = kernel_size // 2  # Automatic padding
        current_filters = 3   # RGB Input
        out_filters = base_filters

        # 2. BUILD THE LAYERS DYNAMICALLY
        for i in range(num_layers):
            block_parts = []

            # A. Convolution
            block_parts.append(nn.Conv2d(current_filters, out_filters,
                                         kernel_size=kernel_size, stride=stride, padding=p))

            # B. Norm & Activation
            block_parts.append(nn.BatchNorm2d(out_filters))
            block_parts.append(nn.ReLU())

            # C. Optional Pooling
            if pool_type == 'max':
                block_parts.append(nn.MaxPool2d(kernel_size=2, stride=2))
            elif pool_type == 'avg':
                block_parts.append(nn.AvgPool2d(kernel_size=2, stride=2))

            # D. Dropout
            if dropout_rate > 0:
                block_parts.append(nn.Dropout(p=dropout_rate))

            self.layers.append(nn.Sequential(*block_parts))

            current_filters = out_filters
            out_filters = out_filters * 2

        # 3. THE "DUMMY PASS" (Shape Calculation)
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, input_size, input_size)
            dummy_out = dummy_input
            for layer in self.layers:
                dummy_out = layer(dummy_out)

            flattened_size = dummy_out.numel()
            if flattened_size == 0:
                raise ValueError("CRITICAL: Image shrunk to 0px! Reduce layers or pooling.")

        # 4. CLASSIFIER
        self.fc = nn.Linear(flattened_size, num_classes)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# ==========================================
# EXPERIMENT EXECUTION (WandB Enabled)
# ==========================================

# 1. DEFINE ALL SETTINGS IN ONE PLACE
experiment_settings = {
    # Structural (Body)
    "base_filters": 16,     # Try 32 or 64 later
    "dropout_rate": 0.0,    # Try 0.2 or 0.5 later
    "kernel_size":  3,      # Try 5 or 7 later
    "num_layers":   3,      # Try 3 (Careful on small images!)
    "stride":       1,
    "pool_type":    'max',

    # Data
    "data_choice": 'original', # 'original' (28px) or 'midsize' (96px)
    "input_size":   28,        # Must match data_choice size!

    # Training (Brain)
    "learning_rate": 0.001,
    "weight_decay":  0.0,
    "epochs":        10
}

EXP_NAME = "Baseline_Run_FullTracking3"

print(f"🏗️ Building {EXP_NAME} with config: {experiment_settings}")

# 2. INSTANTIATE (Unpacking the dictionary)
model = DynamicCNN(
    num_classes=7,
    input_size=experiment_settings["input_size"],
    base_filters=experiment_settings["base_filters"],
    dropout_rate=experiment_settings["dropout_rate"],
    kernel_size=experiment_settings["kernel_size"],
    num_layers=experiment_settings["num_layers"],
    stride=experiment_settings["stride"],
    pool_type=experiment_settings["pool_type"]
)

# 3. TRAIN (Passing the config to WandB)
acc = train_dynamic(
    model,
    EXP_NAME,
    all_loaders,
    epochs=experiment_settings["epochs"],
    loader_key=experiment_settings["data_choice"],
    learning_rate=experiment_settings["learning_rate"],
    weight_decay=experiment_settings["weight_decay"],

    # Send strictural details to WandB
    extra_config=experiment_settings
)

reset_memory(model)

🏗️ Building Baseline_Run_FullTracking3 with config: {'base_filters': 16, 'dropout_rate': 0.0, 'kernel_size': 3, 'num_layers': 3, 'stride': 1, 'pool_type': 'max', 'data_choice': 'original', 'input_size': 28, 'learning_rate': 0.001, 'weight_decay': 0.0, 'epochs': 10}


🚀 Training Baseline_Run_FullTracking3 | LR: 0.001 | Reg: 0.0
   [Epoch 1/10] Loss: 0.8709 | Acc: 0.6922
   [Epoch 2/10] Loss: 0.7406 | Acc: 0.7278
   [Epoch 3/10] Loss: 0.6966 | Acc: 0.7374
   [Epoch 4/10] Loss: 0.6646 | Acc: 0.7530
   [Epoch 5/10] Loss: 0.6330 | Acc: 0.7615
   [Epoch 6/10] Loss: 0.5938 | Acc: 0.7791
   [Epoch 7/10] Loss: 0.5701 | Acc: 0.7911
   [Epoch 8/10] Loss: 0.5327 | Acc: 0.8079
   [Epoch 9/10] Loss: 0.5071 | Acc: 0.8120
   [Epoch 10/10] Loss: 0.4788 | Acc: 0.8273
📝 Validating Baseline_Run_FullTracking3...
🏆 Final Validation Accuracy: 75.27%


train_acc,▁▃▃▄▅▆▆▇▇█
train_loss,█▆▅▄▄▃▃▂▂▁
val_accuracy,▁
train_acc,0.82732
train_loss,0.47876
val_accuracy,0.75274



🧹 Cleaning up GPU memory...
✨ Memory Clean.



In [23]:
## ==========================================
# PHASE 3.2: THE COMPETITOR (ResNet-18)
# ==========================================
from torchvision import models
import torch.nn as nn

def get_resnet_model(num_classes):
    """
    Standard ResNet-18 using Transfer Learning.
    Input requirement: 224x224 images (handled by our new Data Pipeline).
    """
    print("⬇️  Downloading Pre-trained ResNet-18...")

    # 1. Load the "Brain" (Pre-trained on ImageNet)
    # We use 'DEFAULT' weights to get the smartest version available.
    model = models.resnet18(weights='DEFAULT')

    # 2. NO HACK NEEDED!
    # Because we now upscale our images to 224x224 in Phase 2,
    # the original first layer (7x7 conv) works perfectly.
    # We keep it to preserve the pre-trained knowledge.

    # 3. Modify the final classifier
    # ResNet-18 puts 512 features into the final layer.
    # We replace the 1000-class output with our 7 skin classes.
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)

    return model

# --- RUN THE EXPERIMENT ---
print("🏗️ Building ResNet-18 Competitor...")
competitor_model = get_resnet_model(num_classes=n_classes)

# 4. Train using the DYNAMIC Engine
# The engine sees "ResNet" in the name -> Automatically picks 224x224 images
test_acc_resnet = train_dynamic(competitor_model, "ResNet18_Pretrained", all_loaders, epochs=10)
reset_memory(competitor_model) # <--- Wipes GPU again

🏗️ Building ResNet-18 Competitor...
⬇️  Downloading Pre-trained ResNet-18...
🧠 Detected Large Model (ResNet18_Pretrained) -> Switching to 224x224 resolution


🚀 Starting training for ResNet18_Pretrained...
   [Epoch 1/10] Loss: 0.8465 | Acc: 0.7012
   [Epoch 2/10] Loss: 0.6871 | Acc: 0.7393
   [Epoch 3/10] Loss: 0.6413 | Acc: 0.7538
   [Epoch 4/10] Loss: 0.6188 | Acc: 0.7691
   [Epoch 5/10] Loss: 0.5943 | Acc: 0.7738
   [Epoch 6/10] Loss: 0.5600 | Acc: 0.7851
   [Epoch 7/10] Loss: 0.5152 | Acc: 0.8119
   [Epoch 8/10] Loss: 0.4743 | Acc: 0.8148
   [Epoch 9/10] Loss: 0.4156 | Acc: 0.8434
   [Epoch 10/10] Loss: 0.3566 | Acc: 0.8610
📝 Evaluating ResNet18_Pretrained on Test Data...
🏆 Final Test Accuracy for ResNet18_Pretrained: 76.31%


test_accuracy,▁
train_acc,▁▃▃▄▄▅▆▆▇█
train_loss,█▆▅▅▄▄▃▃▂▁
test_accuracy,0.76309
train_acc,0.861
train_loss,0.35656



🧹 Cleaning up GPU memory...
   -> Deleted model object from RAM
   -> Cleared NVIDIA CUDA cache
✨ Memory Clean. Ready for next experiment.



In [24]:
# ==========================================
# PHASE 3.4: THE SPECIALIST (DenseNet-121) - SAFE MODE
# ==========================================
from torchvision import models
import torch.nn as nn
from torch.utils.data import DataLoader

# 1. CREATE SAFE LOADERS (Batch Size 32 instead of 128)
# This prevents the "Out Of Memory" crash by feeding smaller chunks to the GPU
print("📉 Switching to Safe Batch Size (32) for DenseNet...")
loader_safe_train = DataLoader(train_set_large, batch_size=32, shuffle=True)
loader_safe_test  = DataLoader(test_set_large,  batch_size=32, shuffle=False)

# Update the dictionary so the engine uses these new small loaders
all_loaders['large'] = loader_safe_train
all_loaders['test_large'] = loader_safe_test

# 2. BUILD THE MODEL
def get_densenet_model(num_classes):
    print("⬇️  Downloading Pre-trained DenseNet-121...")
    model = models.densenet121(weights='DEFAULT')

    # Modify the head for 7 classes
    num_features = model.classifier.in_features
    model.classifier = nn.Linear(num_features, num_classes)
    return model

print("🏗️ Building DenseNet-121...")
densenet_model = get_densenet_model(n_classes)

# 3. START TRAINING
# The engine will now use the "Safe" loaders we defined above
test_acc_dense = train_dynamic(densenet_model, "DenseNet121", all_loaders, epochs=10)
reset_memory(densenet_model)   # <--- Wipes GPU one last time

📉 Switching to Safe Batch Size (32) for DenseNet...
🏗️ Building DenseNet-121...
⬇️  Downloading Pre-trained DenseNet-121...
🧠 Detected Large Model (DenseNet121) -> Switching to 224x224 resolution


🚀 Starting training for DenseNet121...
   [Epoch 1/10] Loss: 0.8455 | Acc: 0.6943
   [Epoch 2/10] Loss: 0.7484 | Acc: 0.7253
   [Epoch 3/10] Loss: 0.7223 | Acc: 0.7306
   [Epoch 4/10] Loss: 0.6981 | Acc: 0.7408
   [Epoch 5/10] Loss: 0.6641 | Acc: 0.7501
   [Epoch 6/10] Loss: 0.6428 | Acc: 0.7630
   [Epoch 7/10] Loss: 0.6228 | Acc: 0.7691
   [Epoch 8/10] Loss: 0.6152 | Acc: 0.7708
   [Epoch 9/10] Loss: 0.5932 | Acc: 0.7745
   [Epoch 10/10] Loss: 0.5777 | Acc: 0.7814
📝 Evaluating DenseNet121 on Test Data...
🏆 Final Test Accuracy for DenseNet121: 74.91%


test_accuracy,▁
train_acc,▁▃▄▅▅▇▇▇▇█
train_loss,█▅▅▄▃▃▂▂▁▁
test_accuracy,0.74913
train_acc,0.78136
train_loss,0.5777



🧹 Cleaning up GPU memory...
   -> Deleted model object from RAM
   -> Cleared NVIDIA CUDA cache
✨ Memory Clean. Ready for next experiment.



In [ ]:
# ==========================================
# PHASE 3.4: THE SPECIALIST (DenseNet-121) -
# ==========================================
from torchvision import models
import torch.nn as nn
from torch.utils.data import DataLoader

#  1. CREATE SAFE LOADERS (Batch Size 32 instead of 128)
# # This prevents the "Out Of Memory" crash by feeding smaller chunks to the GPU
# print("📉 Switching to Safe Batch Size (32) for DenseNet...")
# loader_safe_train = DataLoader(train_set_large, batch_size=32, shuffle=True)
# loader_safe_test  = DataLoader(test_set_large,  batch_size=32, shuffle=False)

# # Update the dictionary so the engine uses these new small loaders
# all_loaders['large'] = loader_safe_train
# all_loaders['test_large'] = loader_safe_test#

# 2. BUILD THE MODEL
def get_densenet_model(num_classes):
    print("⬇️  Downloading Pre-trained DenseNet-121...")
    model = models.densenet121(weights='DEFAULT')

    # Modify the head for 7 classes
    num_features = model.classifier.in_features
    model.classifier = nn.Linear(num_features, num_classes)
    return model

print("🏗️ Building DenseNet-121...")
densenet_model = get_densenet_model(n_classes)

# 3. START TRAINING
# The engine will now use the "Safe" loaders we defined above
test_acc_dense = train_dynamic(densenet_model, "DenseNetBIG", all_loaders, epochs=10)
reset_memory(densenet_model)   # <--- Wipes GPU one last time